<a href="https://colab.research.google.com/github/gaelosvaldor-star/SheldonRoss/blob/main/Ross.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Simulación vs. Teoría

Modelamos una estación con llegadas Poisson de intensidad $\lambda \ge 0$ y un único servidor. Los clientes esperan en fila si el servidor está ocupado; al liberarse, se atiende al primero en llegar.

Fórmulas teóricas del modelo M/M/1:

- Utilización: $\rho = \frac{\lambda}{\mu}$
- Clientes en cola: $L_q = \frac{\lambda^2}{\mu(\mu-\lambda)}$
- Clientes en sistema: $L = L_q + \frac{\lambda}{\mu}$
- Tiempo en cola: $W_q = \frac{L_q}{\lambda}$
- Tiempo en sistema: $W = W_q + \frac{1}{\mu}$


In [ ]:
import numpy as np
import random as r
from collections import deque

In [ ]:
def simular_cola(T, tasa_llegada, tasa_servicio):

    # Tiempo actual y clientes en sistema
    t = 0.0
    clientes = 0

    # Contadores
    num_arribos = 0
    num_salidas = 0

    # Acumuladores
    area_sist = 0.0
    area_cola = 0.0
    t_ocupado = 0.0
    suma_W = 0.0
    suma_Wq = 0.0
    t_ultimo = 0.0

    fila = deque()

    # Primer arribo
    t_arribo = -(1 / tasa_llegada) * np.log(r.random())
    t_salida = float("inf")
    t_llegada_en_srv = None
    dur_servicio = None

    while True:

        t_sig = min(t_arribo, t_salida)

        # Actualizar áreas mientras el evento esté en el horizonte
        if t_sig <= T:
            dt = t_sig - t_ultimo
            area_sist += clientes * dt
            area_cola += max(clientes - 1, 0) * dt
            if clientes > 0:
                t_ocupado += dt
            t_ultimo = t_sig

        # Caso 1: arribo dentro del horizonte
        if t_arribo < t_salida and t_arribo <= T:
            t = t_arribo
            num_arribos += 1
            clientes += 1
            fila.append(t)
            t_arribo = t - (1 / tasa_llegada) * np.log(r.random())

            if clientes == 1:  # servidor libre
                t_llegada_en_srv = fila.popleft()
                dur_servicio = -(1 / tasa_servicio) * np.log(r.random())
                t_salida = t + dur_servicio

        # Caso 2: salida dentro del horizonte
        elif t_salida < t_arribo and t_salida <= T:
            t = t_salida
            clientes -= 1
            num_salidas += 1
            suma_W  += t - t_llegada_en_srv
            suma_Wq += (t - t_llegada_en_srv) - dur_servicio

            if clientes > 0:
                t_llegada_en_srv = fila.popleft()
                dur_servicio = -(1 / tasa_servicio) * np.log(r.random())
                t_salida = t + dur_servicio
            else:
                t_llegada_en_srv = None
                dur_servicio = None
                t_salida = float("inf")

        # Caso 3: horizonte superado, quedan clientes
        elif min(t_arribo, t_salida) > T and clientes > 0:
            t = t_salida
            clientes -= 1
            num_salidas += 1
            suma_W  += t - t_llegada_en_srv
            suma_Wq += (t - t_llegada_en_srv) - dur_servicio

            if clientes > 0:
                t_llegada_en_srv = fila.popleft()
                dur_servicio = -(1 / tasa_servicio) * np.log(r.random())
                t_salida = t + dur_servicio
            else:
                t_llegada_en_srv = None
                dur_servicio = None
                t_salida = float("inf")

        # Caso 4: fin
        elif min(t_arribo, t_salida) > T and clientes == 0:
            break

    # --- Resultados de simulación ---
    rho_sim = t_ocupado / T
    L_sim   = area_sist / T
    Lq_sim  = area_cola / T
    W_sim   = suma_W / num_salidas
    Wq_sim  = suma_Wq / num_salidas

    # --- Resultados teóricos ---
    rho = tasa_llegada / tasa_servicio
    Lq  = tasa_llegada**2 / (tasa_servicio * (tasa_servicio - tasa_llegada))
    L   = Lq + rho
    Wq  = Lq / tasa_llegada
    W   = Wq + 1 / tasa_servicio

    print("===== SIMULACION =====")
    print(f"rho = {rho_sim:.4f}")
    print(f"Lq  = {Lq_sim:.4f}")
    print(f"L   = {L_sim:.4f}")
    print(f"Wq  = {Wq_sim:.4f}")
    print(f"W   = {W_sim:.4f}")

    print("\n===== TEORICO =====")
    print(f"rho = {rho:.4f}")
    print(f"Lq  = {Lq:.4f}")
    print(f"L   = {L:.4f}")
    print(f"Wq  = {Wq:.4f}")
    print(f"W   = {W:.4f}")


In [ ]:
simular_cola(T=100000, tasa_llegada=4, tasa_servicio=6)


===== SIMULACION =====
rho = 0.6651
Lq  = 1.3265
L   = 1.9916
Wq  = 0.3324
W   = 0.4990

===== TEORICO =====
rho = 0.6667
Lq  = 1.3333
L   = 2.0000
Wq  = 0.3333
W   = 0.5000


Los resultados simulados se aproximan bien a los valores teóricos, lo que valida el modelo.
